# IndicRAG-QA research walkthrough

This notebook compares lexical and multilingual retrieval, then inspects answerability and citation output. The seed corpus contains curated English and Hindi summaries of India's National Education Policy 2020. Treat the small QA set as a software demo, not a final benchmark.

## 1. Load the data

Open this notebook from the repository root so relative data paths resolve. Install the environment with `python -m pip install -r requirements.txt` and select the `Python (IndicRAG-QA)` kernel.

In [ ]:
from indicrag.ingest import load_passages
from indicrag.pipeline import IndicRAG
from indicrag.retrieval import DEFAULT_EMBEDDING_MODEL

passages = load_passages("data/sample/passages.jsonl")
len(passages), passages[0]

## 2. Lexical baseline

BM25 is a term-matching baseline. Compare English, Devanagari Hindi, and Romanized Hinglish; inspect whether the top passage contains the expected evidence.

In [ ]:
engine = IndicRAG(passages, embedding_model=DEFAULT_EMBEDDING_MODEL)
queries = [
    "In which school stage are coding activities introduced?",
    "कोडिंग से जुड़ी गतिविधियाँ किस स्टेज में शुरू होंगी?",
    "NEP me coding activities kaunse stage me hoti hain?",
]

for query in queries:
    print("\nQUERY:", query)
    for hit in engine.retrieve(query, method="bm25", top_k=3):
        print(hit.rank, round(hit.score, 3), hit.passage.passage_id, hit.passage.text)

## 3. Dense and hybrid retrieval

LaBSE is loaded only when dense or hybrid retrieval is called. Its first run downloads model weights. Tune the hybrid lexical weight on the development split; do not choose it from test results.

In [ ]:
query = "NEP me coding activities kaunse school stage me hoti hain?"
for method in ("tfidf", "dense", "hybrid"):
    print("\nMETHOD:", method)
    for hit in engine.retrieve(query, method=method, top_k=3):
        print(hit.rank, round(hit.score, 3), hit.passage.passage_id, hit.component_scores)

## 4. Answer with citations and abstention

The default answerer selects a sentence from retrieved evidence. Calibrate the confidence threshold on dev; the lower threshold here is for demonstrating the response structure only.

In [ ]:
result = engine.ask(query, method="hybrid", top_k=5, threshold=0.10)
print("Query type:", result["query_language"])
print("Answerable:", result["answerable"])
print("Answer:", result["answer"])
for citation in result["citations"][:3]:
    print(citation["passage_id"], citation["source_url"], citation["section"])
    print(citation["excerpt"])

## 5. Evaluate on the development split

Use dev for threshold selection, then apply the selected threshold to test. Full experiments can be run from the terminal with `indicrag evaluate --method bm25 --split test` (repeat with `tfidf`, `dense`, and `hybrid`). Row-level predictions retain retrieved passage IDs for error analysis.

In [ ]:
from indicrag.evaluation import evaluate_dataset

dev_summary, dev_predictions = evaluate_dataset(
    passages,
    "data/eval/qa.jsonl",
    method="bm25",
    split="dev",
    calibrate_on_dev=False,
)
dev_summary

## Next steps

Run all retrieval baselines with the same corpus and K, inspect the language slices and hard cases, then expand the small curated set with human-reviewed questions and evidence annotations before making research claims.